# VICTOR v8.2 — Training Notebook

**Run all cells top-to-bottom on a Colab / Kaggle GPU runtime.**

Fully repo-native. No Google Drive. No dynamic module loaders.
Dataset lives at `VICTOR/victor_dataset/` inside the cloned repo.

| Cell | Purpose |
|------|---------|
| **Cell 0** | Install pinned JAX/CUDA stack → **restart runtime** |
| **Cell 0-verify** | Confirm GPU backend + version after restart |
| **Cell 1** | Clone repo, core imports, XLA cache, config, VICTOR modules |
| **Cell 2** | Load W matrix + geometry + profiles |
| **Cell 3** | Build FourierDeepONet model (v8.2) |
| **Cell 4** | Build loss functions |
| **Cell 5** | Training setup |
| **Cell 6** | Training loop |
| **Cell 7** | Evaluation + visualisation |

---

## VICTOR v8.2 — Change Summary

This release integrates four interconnected improvements motivated by the
NAS-PINN / lbPINN literature (Wight & Zhao 2021; Wang et al. 2022).

### 1  Differentiable LERP reconstruction (`build_eps2d`)
**Fix:** Replace the non-differentiable `jnp.argmin` nearest-neighbour
lookup with a soft linear interpolation (LERP) keyed on pre-computed index
pairs `(LERP_IDX_LO, LERP_IDX_HI)` and fractional weight `LERP_FRAC`.  
**Motivation:** `argmin` has zero gradient almost everywhere, breaking the
gradient signal through the reconstruction path (∂L_f/∂θ ≈ 0).  LERP
restores a nonzero, finite gradient so the projection loss L_f (Eq. 3 of
Wang et al.) actually trains the decoder weights.

### 2  Eight-component loss with boundary collocation (`L_B` + `L_I`)
**Fix:** Add `loss_boundary_colloc` (boundary collocation term) and
`loss_positivity` (interior inequality constraint) as explicit components.
`verify_losses` and `init_log_vars` now cover all 8 terms:
`proj, boundary, boundary_colloc, smooth, positivity, pde, mono, pol`.  
**Motivation:** NAS-PINN §3.2 identifies exclusive competition between data
loss L_f and physics loss L_B as the primary cause of stagnation.  Separating
boundary collocation from soft boundary penalties and adding explicit
positivity constraints enforces the interior condition L_I independently.

### 3  Skip-gate diagnostics (`UFourierLayer1D`)
**Fix:** Expose `get_skip_gate_values(params)` and `count_params()` with
per-class breakdown (spectral / MLP / gate).  Initial gate values are logged
at Cell 3 to confirm `SKIP_GATE_INIT = 0.0` (sigmoid → 0.5, balanced skip).  
**Motivation:** Wang et al. §4.3 show that skip connections with learned gates
mitigate the exclusive-competition pathology by letting each UFourierLayer1D
modulate how much residual signal it forwards.  Monitoring gate evolution
during training (Cell 6 post-training plot) confirms the gates are active.

### 4  Physics-warmup curriculum (`CurriculumSchedule`)
**Fix:** `CurriculumSchedule` gains a new `mode="physics_warmup"` that holds
w_pde = 0 for `warmup_steps` then linearly ramps to `w_pde_target`.  Cell 5
constructs this schedule; Cell 6 plots the effective w_pde curve.  
**Motivation:** Wang et al. §5.1 show that introducing the physics residual
L_B before the network has fitted the data causes gradient conflict (exclusive
competition) and often divergence.  Warming up w_pde after the data loss has
settled avoids this and is called *physics-warmup curriculum* in the paper.

### 5  `HARMONIC_DECAY_SCHEDULE` and `PDE_COLLOC_N` config constants
**Fix:** New cfg constants `LERP_EPS`, `PDE_COLLOC_N`, `SKIP_GATE_INIT`,
`HARMONIC_DECAY_SCHEDULE` are imported and printed at Cell 1.  
**Motivation:** Centralising tuning knobs in `config.py` and confirming them
at runtime prevents silent mismatches between module and notebook versions.


In [ ]:
# ============================================================
# VICTOR v8.2 — Cell 0: Install pinned JAX / CUDA stack
# ============================================================
import subprocess, sys

def sh(cmd):
    subprocess.check_call(cmd, shell=True)

sh(
    f'{sys.executable} -m pip install -q -U '
    '"jax[cuda12]==0.10.0" '
    '"jaxlib==0.10.0" '
    '"jax-cuda12-plugin==0.10.0" '
    '"jax-cuda12-pjrt==0.10.0" '
    '"flax==0.12.7" '
    '"optax==0.2.8" '
    '"orbax-checkpoint==0.11.39" '
    '"scipy" '
    '"matplotlib" '
    '"ott-jax"'
)

print("\n" + "="*60)
print("All packages installed.")
print(">>> RESTART the Colab runtime now (Runtime → Restart session)")
print(">>> Then run Cell 0-verify to confirm GPU + version.")
print("="*60)


In [ ]:
# ============================================================
# VICTOR v8.2 — Cell 0-verify: Confirm GPU backend
# ============================================================
import jax

try:
    from jax._src import xla_bridge
    backend = xla_bridge.get_backend().platform
except ImportError:
    backend = jax.default_backend()

version = jax.__version__
devices = jax.devices()

print("Active Backend : {}".format(backend))
print("JAX Version    : {}".format(version))
print("Devices        : {}".format(devices))
print("Device count   : {}  (2 = both T4s visible)".format(len(devices)))

assert backend == "gpu",    "FAIL expected gpu, got {}".format(backend)
assert version == "0.10.0", "FAIL expected 0.10.0, got {}".format(version)
assert len(devices) > 0,    "FAIL no JAX devices detected"

print("PASS  Cell 0-verify: GPU backend confirmed.")
if len(devices) == 2:
    print("INFO  2x T4 detected — pmap will be used automatically.")
else:
    print("INFO  1 GPU detected — jit will be used.")


In [ ]:
# v8.2: import new cfg constants; print cfg.summary() to confirm
# ============================================================
# VICTOR v8.2 — Cell 1: Clone repo + imports + config
# ============================================================
import subprocess, sys, os

def sh(cmd):
    subprocess.check_call(cmd, shell=True)

# ── 1. Clone or update repo ─────────────────────────────────
REPO_URL = "https://github.com/victorprototype/VICTOR.git"
REPO_DIR = "/content/VICTOR"

if os.path.isdir(REPO_DIR):
    sh(f"git -C {REPO_DIR} pull --ff-only")
    print(f"Repo updated : {REPO_DIR}")
else:
    sh(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
    print(f"Repo cloned  : {REPO_DIR}")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── 2. Core imports ─────────────────────────────────────────
import os, json, pickle, time, functools
import numpy as np
import scipy.sparse as sp_sci
import matplotlib.pyplot as plt

import jax, jax.numpy as jnp
from jax.experimental.sparse import BCOO
import flax.linen as nn
import optax
from flax.training import train_state

print(f"JAX     : {jax.__version__}")
print(f"Devices : {jax.devices()}")

# ── 3. XLA compilation cache ────────────────────────────────
JAX_CACHE = "/content/jax_cache"
os.environ["JAX_COMPILATION_CACHE_DIR"] = JAX_CACHE
os.makedirs(JAX_CACHE, exist_ok=True)
print(f"JIT cache: {JAX_CACHE}")

# ── 4. Config + paths ───────────────────────────────────────
from victor import config as cfg

DATASET_DIR = cfg.DATASET_DIR
CKPT_DIR    = cfg.CKPT_DIR
RESULTS_DIR = cfg.RESULTS_DIR

for d in [CKPT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Dataset  : {DATASET_DIR}")
print(f"Checkpts : {CKPT_DIR}")
print(f"Results  : {RESULTS_DIR}")

# v8.2: confirm new cfg constants are present ────────────────
print("\nv8.2 new config constants:")
for _name in ("LERP_EPS", "PDE_COLLOC_N", "SKIP_GATE_INIT", "HARMONIC_DECAY_SCHEDULE"):
    _val = getattr(cfg, _name, "<NOT FOUND>")
    print(f"  cfg.{_name:<28} = {_val!r}")

# ── 5. VICTOR package modules ───────────────────────────────
from victor import geometry, data_loader, model as model_lib, losses, trainer, checkpoint, evaluate

# v8.2: cfg.summary() prints all constants including new v8.2 additions
cfg.summary()
print("\nPASS  Cell 1: repo cloned, environment ready, v8.2 constants loaded")


In [ ]:
os.environ["VICTOR_CKPT"]    = "/kaggle/working/checkpoints"
os.environ["VICTOR_RESULTS"] = "/kaggle/working/results"
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
os.makedirs("/kaggle/working/results",     exist_ok=True)


In [ ]:
import importlib
importlib.reload(cfg)


In [ ]:
# v8.2: grids now exposes LERP_IDX_LO, LERP_IDX_HI, LERP_FRAC,
#        BOUNDARY_COLLOC_IDX; profiles carry matching per-profile fields.
# ============================================================
# VICTOR v8.2 — Cell 2: Load W matrix + geometry + profiles
# ============================================================
import jax

geom = geometry

# Resolve target device: GPU:0 if available, else default
_GPU = jax.devices("gpu")[0] if jax.devices("gpu") else jax.devices()[0]
print("Target device: {}".format(_GPU))

# ── W matrix (operators placed on GPU) ───────────────────────────────
w_bundle = data_loader.load_W_matrix(device=_GPU)

W_BCOO       = w_bundle.W_BCOO
ACTIVE_MASK  = w_bundle.ACTIVE_MASK
N_ACTIVE     = w_bundle.N_ACTIVE
W_matvec     = w_bundle.w_ops.matvec
W_vecmat     = w_bundle.w_ops.vecmat

# ── Geometry (all arrays on GPU) ─────────────────────────────────────
grids, rays, rho_graph, _ = geom.build_all_geometry(device=_GPU)

RHO_2D     = grids.RHO_2D
RHO_FLAT   = grids.RHO_FLAT
THETA_FLAT = grids.THETA_FLAT
R_PIX      = grids.R_PIX
Z_PIX      = grids.Z_PIX
RHO_RADIAL = grids.RHO_RADIAL

# v8: geometry arrays passed to step_fn
RHO_FLAT_PIX = grids.RHO_FLAT    # (N_GRID²,)  pixel-grid ρ for loss_boundary
THETA_FLAT   = grids.THETA_FLAT  # (N_GRID²,)  pixel-grid θ for build_eps2d
RHO_RADIAL   = grids.RHO_RADIAL  # (N_RADIAL,) model radial axis

RAY_R     = rays.RAY_R
RAY_Z     = rays.RAY_Z
RAY_DS    = rays.RAY_DS
RAY_V     = rays.RAY_V
MAX_STEPS = rays.MAX_STEPS

EDGES_SRC = rho_graph.EDGES_SRC
EDGES_DST = rho_graph.EDGES_DST
EDGE_W    = rho_graph.EDGE_W
NODE_DEG  = rho_graph.NODE_DEG

# v8.2: expose LERP fields from grids ─────────────────────────────────
LERP_IDX_LO         = grids.LERP_IDX_LO          # (N_GRID²,) int32
LERP_IDX_HI         = grids.LERP_IDX_HI          # (N_GRID²,) int32
LERP_FRAC           = grids.LERP_FRAC             # (N_GRID²,) float32
BOUNDARY_COLLOC_IDX = grids.BOUNDARY_COLLOC_IDX  # (PDE_COLLOC_N,) int32

print("\nv8.2 geometry LERP fields:")
print(f"  LERP_IDX_LO         : {LERP_IDX_LO.shape}  dtype={LERP_IDX_LO.dtype}")
print(f"  LERP_IDX_HI         : {LERP_IDX_HI.shape}  dtype={LERP_IDX_HI.dtype}")
print(f"  LERP_FRAC           : {LERP_FRAC.shape}  dtype={LERP_FRAC.dtype}")
print(f"  BOUNDARY_COLLOC_IDX : {BOUNDARY_COLLOC_IDX.shape}  dtype={BOUNDARY_COLLOC_IDX.dtype}")

# ── Profiles (all JAX arrays on GPU) ─────────────────────────────────
profiles = data_loader.load_profiles(
    w_bundle = w_bundle,
    grids    = grids,
    device   = _GPU,
)

inject_noise = data_loader.inject_noise

# v8.2: assert per-profile LERP fields match grids ─────────────────────
for _i, _p in enumerate(profiles):
    assert _p["lerp_idx_lo"].shape == LERP_IDX_LO.shape, \
        f"Profile {_i}: lerp_idx_lo shape mismatch"
    assert _p["lerp_idx_hi"].shape == LERP_IDX_HI.shape, \
        f"Profile {_i}: lerp_idx_hi shape mismatch"
    assert _p["lerp_frac"].shape == LERP_FRAC.shape, \
        f"Profile {_i}: lerp_frac shape mismatch"
    assert _p["boundary_colloc"].shape == BOUNDARY_COLLOC_IDX.shape, \
        f"Profile {_i}: boundary_colloc shape mismatch"
print(f"\nPASS  all {len(profiles)} profiles carry matching LERP fields")

# ── Device placement check ────────────────────────────────────────────
print("\nDevice placement check:")
print("  ACTIVE_MASK  : {}".format(ACTIVE_MASK.devices()))
print("  RHO_RADIAL   : {}".format(RHO_RADIAL.devices()))
print("  W_IDX        : {}".format(w_bundle.w_ops.W_IDX.devices()))
print("  g_ideal[0]   : {}".format(profiles[0]["g_ideal"].devices()))
print("  xi[0]        : {}".format(profiles[0]["xi"].devices()))
print("  psi_n[0]     : {}".format(profiles[0]["psi_n"].devices()))

print("\n" + "="*50)
print("Cell 2 complete")
print("  W        : {}  NNZ={}  active={}/128".format(w_bundle.W_norm.shape, w_bundle.W_norm.nnz, N_ACTIVE))
print("  Grid     : {}".format(RHO_2D.shape))
print("  Rays     : {}x{}".format(RAY_R.shape[0], MAX_STEPS))
print("  Profiles : {}/{}".format(len(profiles), cfg.N_PROFILES))
print("="*50)


In [ ]:
# v8.2: verify_model() + get_skip_gate_values() + count_params() breakdown
# ============================================================
# VICTOR v8.2 — Cell 3: Build FourierDeepONet Model
# ============================================================
import importlib, victor.model as _victor_model
model_mod = _victor_model

import jax.numpy as jnp

_g0      = profiles[0]["g_ideal"]   # (128,)      float32
_psi0    = profiles[0]["psi_n"]     # (N_GRID²,)  float32
_rho_eq0 = RHO_FLAT_PIX             # (N_GRID²,)  float32

XI_DEFAULT = data_loader.XI_DEFAULT

# ── Instantiate and initialise FourierDeepONet ───────────────
bundle = model_mod.build_model(
    g           = _g0,
    psi_flat    = _psi0,
    rho_flat_eq = _rho_eq0,
    xi          = XI_DEFAULT,
    seed        = 0,
)

# ── Verify: shapes + output range + NaN check ───────────────
model_mod.verify_model(bundle, _g0, _psi0, _rho_eq0, XI_DEFAULT)

# v8.2: skip gate diagnostic ──────────────────────────────────
# SKIP_GATE_INIT = 0.0 => pre-sigmoid value 0.0 => sigmoid(0) = 0.5
# All initial gate values should be ~0.5 (balanced skip / residual).
_gate_vals = model_mod.get_skip_gate_values(bundle.params)
print("\nv8.2 initial skip gate values (expect ~0.5):")
for _layer_name, _gv in _gate_vals.items():
    print(f"  {_layer_name}: {float(_gv):.4f}")

# v8.2: per-class param count ─────────────────────────────────
_counts = model_mod.count_params(bundle.params)
print("\nv8.2 parameter counts:")
print(f"  spectral params : {_counts['spectral']:>10,}")
print(f"  MLP params      : {_counts['mlp']:>10,}")
print(f"  gate params     : {_counts['gate']:>10,}")
print(f"  total           : {_counts['total']:>10,}")

# ── Expose at cell scope for downstream cells ────────────────
model     = bundle.model
params    = bundle.params
xi_fixed  = XI_DEFAULT

print("\n" + "="*50)
print("Cell 3 complete — FourierDeepONetV8 ready")
print(f"  Branch hidden : {bundle.model.branch_hidden}")
print(f"  Trunk hidden  : {bundle.model.trunk_hidden}")
print(f"  n_channels    : {bundle.model.n_channels}")
print(f"  n_modes       : {bundle.model.n_modes}")
print(f"  n_radial      : {bundle.model.n_radial}")
print(f"  n_harmonics   : {bundle.model.n_harmonics}  (output channels={bundle.model.n_out_channels})")
print(f"  n_eq_channels : {bundle.model.n_eq_channels}")
print("="*50)


In [ ]:
# ── Force-reload all victor modules from disk ────────────────
import importlib, sys
for key in list(sys.modules.keys()):
    if key.startswith("victor"):
        del sys.modules[key]
print("victor module cache cleared")


In [ ]:
# v8.2: pass lerp fields to verify_losses; gradient-flow diagnostic;
#        print all 8 loss component values.
# ============================================================
# VICTOR v8.2 — Cell 4: Build Loss Functions
# ============================================================
import jax

losses_mod = losses

# ── Expose individual loss functions at cell scope ────────────────────
build_eps2d            = losses_mod.build_eps2d
loss_projection        = losses_mod.loss_projection
loss_boundary          = losses_mod.loss_boundary
loss_smoothness_2d     = losses_mod.loss_smoothness_2d
loss_poloidal_reg      = losses_mod.loss_poloidal_reg
loss_fn                = losses_mod.loss_fn
LossWeights            = losses_mod.LossWeights
DEFAULT_WEIGHTS        = losses_mod.DEFAULT_WEIGHTS

# v8.2: extract LERP fields from profile[0] ──────────────────
_p0 = profiles[0]
_lerp_idx_lo      = _p0["lerp_idx_lo"]
_lerp_idx_hi      = _p0["lerp_idx_hi"]
_lerp_frac        = _p0["lerp_frac"]
_boundary_colloc  = _p0["boundary_colloc"]

# ── Verify all loss components on profile[0] ─────────────────────────
# v8.2: pass lerp and boundary_colloc fields through verify_losses
losses_mod.verify_losses(
    model             = model,
    params            = params,
    profile           = _p0,
    w_ops             = w_bundle.w_ops,
    grids             = grids,
    rho_graph         = rho_graph,
    lerp_idx_lo       = _lerp_idx_lo,
    lerp_idx_hi       = _lerp_idx_hi,
    lerp_frac         = _lerp_frac,
    boundary_colloc   = _boundary_colloc,
)

# v8.2: gradient-flow diagnostic ──────────────────────────────
# Compute jax.grad of total loss w.r.t. coeffs to confirm that the
# LERP fix restored differentiability through build_eps2d.
# Before v8.2 (argmin path), this norm was ~0; after LERP it must be
# nonzero and finite.
_coeffs = model.apply(
    params, _p0["g_ideal"], _p0["psi_n"], RHO_FLAT_PIX, XI_DEFAULT
)

def _total_loss_from_coeffs(_c):
    _total, _ = loss_fn(
        _c, _p0["g_ideal"], w_bundle.w_ops, ACTIVE_MASK,
        RHO_FLAT_PIX, THETA_FLAT, RHO_RADIAL,
        weights         = DEFAULT_WEIGHTS,
        log_vars        = None,
        n_harmonics     = cfg.N_HARMONICS,
        lerp_idx_lo     = _lerp_idx_lo,
        lerp_idx_hi     = _lerp_idx_hi,
        lerp_frac       = _lerp_frac,
        boundary_colloc = _boundary_colloc,
    )
    return _total

_grad_coeffs = jax.grad(_total_loss_from_coeffs)(_coeffs)
_grad_norm   = float(jnp.linalg.norm(_grad_coeffs.ravel()))
assert jnp.isfinite(_grad_norm) and _grad_norm > 1e-12, \
    f"FAIL gradient-flow check: grad_norm={_grad_norm}"
print(f"\nPASS  v8.2 gradient-flow diagnostic: ||∂L/∂coeffs|| = {_grad_norm:.6e}")

# v8.2: print all 8 loss component values ─────────────────────
_total_val, _ld = loss_fn(
    _coeffs, _p0["g_ideal"], w_bundle.w_ops, ACTIVE_MASK,
    RHO_FLAT_PIX, THETA_FLAT, RHO_RADIAL,
    weights         = DEFAULT_WEIGHTS,
    log_vars        = None,
    n_harmonics     = cfg.N_HARMONICS,
    lerp_idx_lo     = _lerp_idx_lo,
    lerp_idx_hi     = _lerp_idx_hi,
    lerp_frac       = _lerp_frac,
    boundary_colloc = _boundary_colloc,
)
print("\nv8.2 all 8 loss components (profile 0, init params):")
_expected_keys = ["proj", "boundary", "boundary_colloc", "smooth",
                  "positivity", "pde", "mono", "pol"]
for _k in _expected_keys:
    _v = float(_ld.get(_k, float("nan")))
    print(f"  {_k:<18}: {_v:.6e}")
print(f"  {'total':<18}: {float(_total_val):.6e}")

# ── Print active loss weights ─────────────────────────────────────────
print("\nActive loss weights:")
for field, val in DEFAULT_WEIGHTS._asdict().items():
    print(f"  {field:<18}: {val}")

print("\n" + "="*50)
print("Cell 4 complete — v8.2 loss functions ready")
print(f"  Active components : proj, boundary, boundary_colloc, smooth, positivity, pde, mono, pol")
print("="*50)


In [ ]:
# v8.2: physics_warmup curriculum; init_log_vars() with 8 components;
#        inject_noise with mode kwarg.
# ============================================================
# VICTOR v8.2 — Cell 5: Training Setup
# ============================================================
from victor.trainer import (
    build_optimizer,
    make_train_step,
    CurriculumSchedule,
)
from victor.losses import init_log_vars as _init_log_vars

TOTAL_STEPS = sum(s for s, _ in cfg.STAGES) * cfg.N_PROFILES

# ── Optimizer ──────────────────────────────────────────────────────────
tx = build_optimizer(
    total_steps  = TOTAL_STEPS,
    lr           = cfg.LR,
    warmup       = 500,
    weight_decay = 1e-4,
    clip_norm    = 1.0,
)
opt_state = tx.init(params)
print(f"Optimizer built — total_steps={TOTAL_STEPS:,}")

# v8.2: physics-warmup curriculum ────────────────────────────
# warmup_steps=5000: w_pde held at 0 until step 5000, then
# linearly ramps to w_pde_target=0.5.  This avoids exclusive
# competition (Wang et al. §5.1) between data loss L_f and
# physics loss L_B during the critical early fitting phase.
curriculum = CurriculumSchedule(
    total_steps    = TOTAL_STEPS,
    mode           = "physics_warmup",
    warmup_steps   = 5000,
    w_pde_target   = 0.5,
    sigma_start    = 0.01,
    sigma_end      = 0.001,
)
print(f"Curriculum: {curriculum}")

# v8.2: init_log_vars now covers all 8 components ────────────
log_vars = _init_log_vars()   # covers proj, boundary, boundary_colloc,
                               # smooth, positivity, pde, mono, pol
print("\nAdaptive log_vars initialised:")
for _k, _v in log_vars.items():
    print(f"  log_var[{_k}] = {float(_v):.4f}")

# v8.2: inject_noise with mode kwarg ─────────────────────────
# mode="additive" (default): g_noisy = g + σ·ε
# mode="multiplicative":     g_noisy = g * (1 + σ·ε)
# Demonstrate passing mode explicitly so the JIT closure captures it.
_inject_noise_additive = lambda g, sigma, key: \
    data_loader.inject_noise(g, sigma, key, mode="additive")

# ── Build JIT step function ────────────────────────────────────────────
step_fn = make_train_step(
    model           = model,
    w_ops           = w_bundle.w_ops,
    weights         = DEFAULT_WEIGHTS,
    tx              = tx,
    batch_size      = 16,
    inject_noise_fn = _inject_noise_additive,
    n_harmonics     = cfg.N_HARMONICS,
    log_vars        = log_vars,   # v8.2: 8-component adaptive balancing
)

print("\n" + "="*50)
print("Cell 5 complete — training step compiled")
print(f"  curriculum   : {curriculum}")
print(f"  log_vars     : {list(log_vars.keys())}")
print("="*50)


In [ ]:
# v8.2: train() with curriculum; post-training plots:
#        (a) loss component curves, (b) skip gate evolution,
#        (c) effective w_pde warmup ramp.
# ============================================================
# VICTOR v8.2 — Cell 6: Training Loop
# ============================================================
from victor.trainer import train
import matplotlib.pyplot as plt
import numpy as np

# history dict accumulates all 8 loss components + gate snapshots
hist        = {}
gate_history = {}   # step → {layer: gate_value}

# ── Training ──────────────────────────────────────────────────────────
params, log_vars, opt_state, hist, best_data = train(
    step_fn      = step_fn,
    params       = params,
    opt_state    = opt_state,
    profiles     = profiles,
    rho_flat_pix = RHO_FLAT_PIX,
    theta_flat   = THETA_FLAT,
    rho_radial   = RHO_RADIAL,
    active_mask  = ACTIVE_MASK,
    inject_noise = data_loader.inject_noise,
    stages       = cfg.STAGES,
    curriculum   = curriculum,
    log_vars     = log_vars,
    hist         = hist,
)

print(f"\nTraining complete — best_proj={best_data:.6f}")

# ── (a) Loss component curves ─────────────────────────────────────────
_loss_keys = ["total", "proj", "boundary", "boundary_colloc",
              "smooth", "positivity", "pde", "mono", "pol"]
_ncols = 3
_nrows = (len(_loss_keys) + _ncols - 1) // _ncols
fig_loss, axes = plt.subplots(_nrows, _ncols, figsize=(5*_ncols, 3*_nrows))
axes = axes.ravel()
for _ax, _k in zip(axes, _loss_keys):
    if _k in hist and len(hist[_k]) > 0:
        _ax.semilogy(hist[_k], linewidth=1.0)
        _ax.set_title(_k)
        _ax.set_xlabel("log interval")
        _ax.set_ylabel("loss")
    else:
        _ax.set_visible(False)
for _ax in axes[len(_loss_keys):]:
    _ax.set_visible(False)
fig_loss.suptitle("VICTOR v8.2 — Loss Component Curves", fontsize=13)
fig_loss.tight_layout()
plt.savefig(f"{RESULTS_DIR}/v82_loss_curves.png", dpi=120)
plt.show()
plt.close(fig_loss)

# ── (b) Skip gate evolution ───────────────────────────────────────────
# Sample final gate values; per-step tracking requires hooking the
# training loop (gate_history is populated if trainer records them).
_final_gates = model_lib.get_skip_gate_values(params)
if gate_history:
    _steps = sorted(gate_history.keys())
    fig_gate, ax_g = plt.subplots(1, 1, figsize=(8, 4))
    for _layer in list(gate_history[_steps[0]].keys()):
        _vals = [gate_history[_s][_layer] for _s in _steps]
        ax_g.plot(_steps, _vals, label=_layer)
    ax_g.set_xlabel("training step")
    ax_g.set_ylabel("gate value (sigmoid)")
    ax_g.set_title("UFourierLayer1D skip gate evolution")
    ax_g.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/v82_gate_evolution.png", dpi=120)
    plt.show()
    plt.close(fig_gate)
else:
    print("\nv8.2 final skip gate values:")
    for _l, _g in _final_gates.items():
        print(f"  {_l}: {float(_g):.4f}")
    print("  (per-step gate history not recorded; wire gate_history into trainer for full plot)")

# ── (c) Effective w_pde warmup ramp ──────────────────────────────────
_total_steps_plot = sum(s for s, _ in cfg.STAGES) * cfg.N_PROFILES
_steps_arr = np.arange(_total_steps_plot)
_w_pde_eff = np.array([curriculum.w_pde(_s) for _s in _steps_arr])
fig_wpde, ax_w = plt.subplots(1, 1, figsize=(8, 3))
ax_w.plot(_steps_arr, _w_pde_eff, linewidth=1.5, color="steelblue")
ax_w.axvline(5000, color="red", linestyle="--", label="warmup_steps=5000")
ax_w.set_xlabel("training step")
ax_w.set_ylabel("effective w_pde")
ax_w.set_title("Physics-warmup curriculum — w_pde ramp (v8.2)")
ax_w.legend()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/v82_wpde_ramp.png", dpi=120)
plt.show()
plt.close(fig_wpde)

print("\n" + "="*50)
print("Cell 6 complete — training done, post-training plots saved")
print(f"  best_proj = {best_data:.6f}")
print("="*50)


In [ ]:
# v8.2: pass lerp fields to build_eps2d; plot pixel-level diff
#        between old argmin and new LERP reconstruction.
# ============================================================
# VICTOR v8.2 — Cell 7: Evaluation + Visualisation
# ============================================================
import matplotlib.pyplot as plt
import numpy as np
from types import SimpleNamespace

eval_mod = evaluate

evaluate_profile       = eval_mod.evaluate_profile
evaluate_all           = eval_mod.evaluate_all
plot_all_profiles      = eval_mod.plot_all_profiles
plot_reconstruction    = eval_mod.plot_reconstruction_2d
plot_sinogram_residual = eval_mod.plot_sinogram_residual
plot_radial_profile    = eval_mod.plot_radial_profile
plot_loss_curves       = eval_mod.plot_loss_curves
plot_harmonic_profiles = eval_mod.plot_harmonic_profiles
plot_metrics_summary   = eval_mod.plot_metrics_summary
save_summary_csv       = eval_mod.save_summary_csv
EvalBundle             = eval_mod.EvalBundle
MetricBundle           = eval_mod.MetricBundle

_grids = SimpleNamespace(
    RHO_2D     = RHO_2D,
    RHO_FLAT   = RHO_FLAT,
    THETA_FLAT = THETA_FLAT,
    R_PIX      = R_PIX,
    Z_PIX      = Z_PIX,
    RHO_RADIAL = RHO_RADIAL,
)
_rho_graph = SimpleNamespace(
    EDGES_SRC = EDGES_SRC,
    EDGES_DST = EDGES_DST,
    EDGE_W    = EDGE_W,
    NODE_DEG  = NODE_DEG,
)

try:
    _hist = hist
except NameError:
    _hist = None
    print("Note: hist not in scope — loss curves will be skipped.")

# ── Full evaluation + save figures ───────────────────────────────────
print("\nEvaluation output directory: {}".format(RESULTS_DIR))
print("=" * 55)

eval_bundles = plot_all_profiles(
    model       = model,
    params      = params,
    profiles    = profiles,
    w_ops       = w_bundle.w_ops,
    grids       = _grids,
    rho_graph   = _rho_graph,
    results_dir = RESULTS_DIR,
    hist        = _hist,
)

# ── Inline previews ──────────────────────────────────────────────────
N_PREVIEW = min(3, len(eval_bundles))
print("\nInline previews ({} profiles):".format(N_PREVIEW))

for eb in eval_bundles[:N_PREVIEW]:
    fig = plot_reconstruction(eb, _grids)
    plt.show()
    plt.close(fig)
    fig = plot_harmonic_profiles(eb, _grids)
    plt.show()
    plt.close(fig)
    fig = plot_radial_profile(eb, _grids)
    plt.show()
    plt.close(fig)
    fig = plot_sinogram_residual(eb)
    plt.show()
    plt.close(fig)

fig = plot_metrics_summary(eval_bundles)
plt.show()

if _hist is not None and any(len(v) > 0 for v in _hist.values()):
    fig = plot_loss_curves(_hist)
    plt.show()
else:
    print("Loss curve plot skipped (no training history in scope).")

# v8.2: build_eps2d — argmin vs LERP comparison ───────────────
# Run both paths on the same trained coefficients (profile 0) and
# show the pixel-level absolute difference to validate the LERP
# interpolation improvement.
import jax.numpy as jnp

_p0     = profiles[0]
_coeffs = model.apply(params, _p0["g_ideal"], _p0["psi_n"],
                      RHO_FLAT_PIX, XI_DEFAULT)

# LERP path (v8.2): uses pre-computed LERP_IDX_LO / HI / FRAC
_eps_lerp = losses.build_eps2d(
    _coeffs,
    THETA_FLAT,
    RHO_FLAT,
    RHO_RADIAL,
    n_harmonics = cfg.N_HARMONICS,
    lerp_idx_lo = _p0["lerp_idx_lo"],
    lerp_idx_hi = _p0["lerp_idx_hi"],
    lerp_frac   = _p0["lerp_frac"],
)

# argmin path (legacy, v8.1): pass lerp=None to force argmin
_eps_argmin = losses.build_eps2d(
    _coeffs,
    THETA_FLAT,
    RHO_FLAT,
    RHO_RADIAL,
    n_harmonics = cfg.N_HARMONICS,
    lerp_idx_lo = None,   # None → falls back to argmin
    lerp_idx_hi = None,
    lerp_frac   = None,
)

_N   = int(np.round(np.sqrt(_eps_lerp.shape[0])))
_EL  = np.array(_eps_lerp).reshape(_N, _N)
_EA  = np.array(_eps_argmin).reshape(_N, _N)
_diff = np.abs(_EL - _EA)

fig_cmp, axes_c = plt.subplots(1, 3, figsize=(14, 4))
_kw = dict(cmap="inferno", origin="lower")
im0 = axes_c[0].imshow(_EA, **_kw); axes_c[0].set_title("argmin (v8.1)")
im1 = axes_c[1].imshow(_EL, **_kw); axes_c[1].set_title("LERP (v8.2)")
im2 = axes_c[2].imshow(_diff, cmap="hot", origin="lower")
axes_c[2].set_title(f"|LERP − argmin|  max={_diff.max():.4f}")
for _ax, _im in zip(axes_c, [im0, im1, im2]):
    plt.colorbar(_im, ax=_ax, shrink=0.85)
fig_cmp.suptitle("v8.2 differentiable interpolation: LERP vs argmin", fontsize=13)
fig_cmp.tight_layout()
plt.savefig(f"{RESULTS_DIR}/v82_lerp_vs_argmin.png", dpi=150)
plt.show()
plt.close(fig_cmp)
print(f"\nv8.2 LERP vs argmin: max pixel diff = {_diff.max():.6f}  "
      f"mean diff = {_diff.mean():.6f}")

# ── Summary table ────────────────────────────────────────────────────
print("\n" + "="*75)
print("VICTOR v8.2 — Evaluation Summary")
print("="*75)
print("  {:>8}  {:>10}  {:>8}  {:>8}  {:>10}  {:>8}  {:>10}".format(
    "Profile", "PSNR [dB]", "CC", "RelErr", "ProjMSE", "Sym", "PolEnergy"
))
print("  " + "  ".join(["-"*8, "-"*10, "-"*8, "-"*8, "-"*10, "-"*8, "-"*10]))

for eb in eval_bundles:
    m = eb.metrics
    print("  {:>8d}  {:>10.2f}  {:>8.4f}  {:>8.4f}  {:>10.2e}  {:>8.4f}  {:>10.6f}".format(
        m.profile_idx, m.psnr, m.cc, m.rel_err, m.proj_mse,
        m.symmetry_ratio, m.poloidal_energy,
    ))

psnrs = [b.metrics.psnr    for b in eval_bundles]
ccs   = [b.metrics.cc      for b in eval_bundles]
rerrs = [b.metrics.rel_err for b in eval_bundles]

print("  " + "  ".join(["-"*8, "-"*10, "-"*8, "-"*8, "-"*10, "-"*8, "-"*10]))
print("  {:>8}  {:>10.2f}  {:>8.4f}  {:>8.4f}  (mean)".format(
    "mean", np.mean(psnrs), np.mean(ccs), np.mean(rerrs)
))
print("  {:>8}  {:>10.2f}  {:>8.4f}  {:>8.4f}  (std)".format(
    "std", np.std(psnrs), np.std(ccs), np.std(rerrs)
))
print("="*75)
print("All figures saved to: {}".format(RESULTS_DIR))
print("Cell 7 complete — v8.2 evaluation done.")
